# Notebook 5 (V9) — Sample Path with Regime Switch

Simulate a single history of the regime process from $z_0=u$:
- stay in $u$ with probability $\pi$;
- switch to $b$ with probability $1-\pi$;
- once in $b$, remain in $b$ forever.

Using the policy maps from notebook 3 (u-branch) and notebook 2 (BGP), construct the path of all variables across the regime switch. The expected V9 result: the per-variety price-dividend ratio rises along the u-branch, then becomes constant after the switch — the bubble bursts when the regime absorbs.

In [ ]:
using Pkg; Pkg.activate(".")
include("TwoCountryProductionOLG.jl")
using Plots, LaTeXStrings, Printf, Random
gr()

In [ ]:
p = ProductionParams(T_max=30, common_world_growth=true, branch_iters=20)
result = run_production_simulation(p; verbose=true)

## 1. Construct a Sample Path with a Specified Switch Date

In [ ]:
# Switch date τ: economy is in u for t < τ, in b for t ≥ τ
function build_switch_path(p::ProductionParams,
                            u_path::Vector{UPeriodState},
                            bgp_seq::Vector{BGPResult},
                            τ::Int)
    T = length(u_path)
    @assert 1 ≤ τ ≤ T+1
    state = NamedTuple[]
    for t in 1:T
        if t < τ
            s = u_path[t]
            push!(state, (t=t, regime=:u,
                          φ_US=s.φ_US, φ_W=s.φ_W,
                          q_US=s.q_US, d_US=s.d_US,
                          q_W=s.q_W, d_W=s.d_W,
                          Q_US=s.Q_US, Q_W=s.Q_W,
                          e_US=s.e_US, e_W=s.e_W,
                          Y_US=s.Y_US, Y_W=s.Y_W,
                          N_US=s.N_US, N_W=s.N_W,
                          ω=s.ω, ω_star=s.ω_star,
                          θ=s.θ, θ_US_star=s.θ_US_star,
                          R_f=s.R_f, R_f_W=s.R_f_W))
        else
            # Use BGP at the predetermined state at date t
            # (Knowledge stocks: N_US,t was determined by u-history pre-switch,
            #  then evolves at G_N_US^b after switch.)
            bgp = bgp_seq[t]
            push!(state, (t=t, regime=:b,
                          φ_US=bgp.φ_US, φ_W=bgp.φ_W,
                          q_US=bgp.q_US, d_US=bgp.d_US,
                          q_W=bgp.q_W, d_W=bgp.d_W,
                          Q_US=bgp.Q_US, Q_W=bgp.Q_W,
                          e_US=bgp.e_US, e_W=bgp.e_W,
                          Y_US=bgp.Y_US, Y_W=bgp.Y_W,
                          N_US=bgp.N_US, N_W=bgp.N_W,
                          ω=bgp.ω, ω_star=bgp.ω_star,
                          θ=bgp.θ, θ_US_star=bgp.θ_US_star,
                          R_f=bgp.R_f, R_f_W=bgp.R_f_W))
        end
    end
    return state
end

τ_switch = 12   # switch happens at t = 12
sim = build_switch_path(p, result.u_path, result.bgp_seq, τ_switch)

println("Switch date τ = $τ_switch (period $(τ_switch-1) is the last u-period)")
println("Simulation horizon: $(p.T_max) periods")

## 2. Plot the V9 Transition: Per-Variety Price-Dividend Ratio Across the Switch

In [ ]:
T = length(sim)
tt = 1:T

qd_path  = [s.q_US/s.d_US for s in sim]
QD_path  = [s.Q_US/(s.N_US*s.d_US) for s in sim]
φ_path   = [s.φ_US for s in sim]
Y_path   = [s.Y_US for s in sim]
rel_path = [s.Y_W/s.Y_US for s in sim]
regime_z = [s.regime for s in sim]

switch_x = τ_switch - 0.5

p1 = plot(tt, qd_path, lw=2, marker=:circle, label=L"q_{US,t}/d_{US,t}",
          xlabel="period t", ylabel="price-dividend ratio",
          title="Per-variety US price-dividend ratio (V9: bubble grows then collapses)")
vline!(p1, [switch_x], ls=:dash, color=:red, label="switch τ = $τ_switch")

p2 = plot(tt, φ_path, lw=2, marker=:circle, label=L"\varphi_{US,t}",
          xlabel="period t", ylabel=L"\varphi",
          title="US labour allocation")
vline!(p2, [switch_x], ls=:dash, color=:red, label="")

p3 = plot(tt, Y_path, lw=2, marker=:circle, label=L"Y_{US,t}",
          xlabel="period t", ylabel="output (log)", yscale=:log10,
          title="US output")
vline!(p3, [switch_x], ls=:dash, color=:red, label="")

p4 = plot(tt, rel_path, lw=2, marker=:circle, label=L"Y_W/Y_{US}",
          xlabel="period t", ylabel="ratio",
          title="Relative country size")
vline!(p4, [switch_x], ls=:dash, color=:red, label="")

plot(p1, p2, p3, p4, layout=(2,2), size=(1000, 750))

## 3. Portfolio Policies Across the Switch

In [ ]:
ω_path   = [s.ω         for s in sim]
ωs_path  = [s.ω_star    for s in sim]
θ_path   = [s.θ         for s in sim]
θUs_path = [s.θ_US_star for s in sim]
Rf_path  = [s.R_f       for s in sim]
RfW_path = [s.R_f_W     for s in sim]

p1 = plot(tt, ω_path, lw=2, marker=:circle, label=L"\omega_t",
          xlabel="period t", ylabel="weight",
          title="Equity portfolio weights")
plot!(p1, tt, ωs_path, lw=2, marker=:square, label=L"\omega_t^*")
vline!(p1, [switch_x], ls=:dash, color=:red, label="")

p2 = plot(tt, θ_path, lw=2, marker=:circle, label=L"\theta_t",
          xlabel="period t", ylabel="bond share",
          title="Bond shares")
plot!(p2, tt, θUs_path, lw=2, marker=:square, label=L"\theta_{US,t}^*")
hline!(p2, [0.0], ls=:dash, color=:black, label="")
vline!(p2, [switch_x], ls=:dash, color=:red, label="")

p3 = plot(tt, Rf_path, lw=2, marker=:circle, label=L"R_{f,t}",
          xlabel="period t", ylabel="return",
          title="Risk-free rates")
plot!(p3, tt, RfW_path, lw=2, marker=:square, label=L"R_{f,t}^W")
vline!(p3, [switch_x], ls=:dash, color=:red, label="")

p4 = plot(tt, RfW_path .- Rf_path, lw=2, marker=:circle, label=L"R_f^W - R_f",
          xlabel="period t", ylabel="spread (exorbitant privilege)",
          title="US bond convenience yield")
hline!(p4, [0.0], ls=:dash, color=:black, label="")
vline!(p4, [switch_x], ls=:dash, color=:red, label="")

plot(p1, p2, p3, p4, layout=(2,2), size=(1000, 750))

## 4. Stochastic Sample Path (Monte Carlo)

In [ ]:
Random.seed!(42)

function simulate_one_path(p::ProductionParams, result, T::Int=p.T_max)
    z = :u
    τ = T + 1
    for t in 1:T
        if z == :u && rand() > p.π_persist
            τ = t; z = :b; break
        end
    end
    return build_switch_path(p, result.u_path, result.bgp_seq, τ), τ
end

# 5 sample paths
paths = Vector{Vector{NamedTuple}}(); switches = Int[]
for _ in 1:5
    sim, τ = simulate_one_path(p, result)
    push!(paths, sim); push!(switches, τ)
end

println("Sample switch dates: ", switches)

p1 = plot(xlabel="period t", ylabel=L"q_{US}/d_{US}",
          title="Per-variety price-dividend ratio: 5 stochastic sample paths")
for (i, sim) in enumerate(paths)
    qd = [s.q_US/s.d_US for s in sim]
    τ = switches[i]
    plot!(p1, 1:length(sim), qd, lw=2, marker=:circle, ms=3,
          label="τ=$τ")
end
p1

## 5. Save Simulation to CSV

In [ ]:
open("v9_sim_τ$(τ_switch).csv", "w") do io
    write(io, "t,regime,phi_US,phi_W,q_US,d_US,q_W,d_W,Q_US,Q_W,e_US,e_W,Y_US,Y_W,N_US,N_W,omega,omega_star,theta,theta_US_star,R_f,R_f_W\n")
    for s in sim
        write(io, "$(s.t),$(s.regime),$(s.φ_US),$(s.φ_W),$(s.q_US),$(s.d_US),$(s.q_W),$(s.d_W),$(s.Q_US),$(s.Q_W),$(s.e_US),$(s.e_W),$(s.Y_US),$(s.Y_W),$(s.N_US),$(s.N_W),$(s.ω),$(s.ω_star),$(s.θ),$(s.θ_US_star),$(s.R_f),$(s.R_f_W)\n")
    end
end
println("Saved v9_sim_τ$(τ_switch).csv")

## Summary

- The deterministic switch path shows the V9 mechanism: $q^u/d^u$ rises along the u-branch and stabilises after the switch.
- Random draws of the regime process show the heterogeneity in switch dates and the resulting price-dividend trajectories.
- All simulated quantities are saved to a CSV for downstream analysis.